# Notebook 4 — MATLAB `struct` to Python `dict` for ROS 2

## Purpose

This notebook helps MATLAB users understand Python dictionaries through a familiar comparison:

- MATLAB `struct`
- Python `dict`
- JSON files
- CSV files
- `pathlib` file paths
- nested robot configuration
- ROS 2 parameter-style data

The notebook uses only the Python standard library, so it can run without ROS 2 installed.

---

## Learning objectives

By the end of this notebook, you should be able to:

1. Create and access Python dictionaries.
2. Compare MATLAB structures with Python dictionaries.
3. Modify, add, and remove dictionary entries.
4. Work with nested dictionaries.
5. Iterate through keys and values.
6. Save dictionaries as JSON.
7. Load JSON files back into Python.
8. read and write CSV files.
9. use `pathlib` for file and folder operations.
10. recognize how dictionary-shaped data appears in ROS 2 configurations.


## 1. MATLAB structure and Python dictionary

A MATLAB structure stores named fields.

```matlab
robot.name = "RR robot";
robot.mass = 5.0;
robot.link_length = [0.5 0.4];
```

A Python dictionary stores named keys and associated values.


In [ ]:
robot = {
    "name": "RR robot",
    "mass": 5.0,
    "link_length": [0.5, 0.4]
}

print(robot)


### Direct comparison

| Engineering idea | MATLAB | Python |
|---|---|---|
| Create structure | `robot.mass = 5.0` | `robot["mass"] = 5.0` |
| Read value | `robot.mass` | `robot["mass"]` |
| Field/key names | `fieldnames(robot)` | `robot.keys()` |
| Remove field/key | `rmfield(robot,"mass")` | `robot.pop("mass")` |
| Nested data | `robot.controller.kp` | `robot["controller"]["kp"]` |

A Python dictionary is not exactly the same as a MATLAB structure, but it is the closest basic comparison.


## 2. Access dictionary values

In [ ]:
print(robot["name"])
print(robot["mass"])
print(robot["link_length"])


Dictionary access uses square brackets with a key:

```python
robot["mass"]
```

This differs from list access:

```python
joint_values[0]
```

- A list normally uses an integer index.
- A dictionary normally uses a meaningful key.


## 3. Add and modify entries

In [ ]:
robot["manufacturer"] = "Teaching Laboratory"
robot["mass"] = 5.5

print(robot)


The same syntax is used for both operations:

- if the key exists, its value is replaced;
- if the key does not exist, a new key-value pair is created.


## 4. Safe access with `get()`

In [ ]:
print(robot.get("mass"))
print(robot.get("payload"))
print(robot.get("payload", 0.0))


Using square brackets with a missing key raises a `KeyError`.

Using `get()` allows a default value:

```python
payload = robot.get("payload", 0.0)
```

This is useful when configuration values are optional.


## 5. Keys, values, and items

In [ ]:
print("Keys:")
print(robot.keys())

print("\nValues:")
print(robot.values())

print("\nKey-value pairs:")
print(robot.items())


### Loop through a dictionary

In [ ]:
for key, value in robot.items():
    print(f"{key}: {value}")


This pattern is common in configuration handling, logging, and parameter inspection.


## 6. Nested dictionaries

In [ ]:
robot_config = {
    "robot": {
        "name": "RR robot",
        "number_of_joints": 2,
        "link_length": [0.5, 0.4]
    },
    "controller": {
        "kp": [100.0, 100.0],
        "kd": [20.0, 20.0],
        "sampling_time": 0.001
    },
    "simulation": {
        "duration": 10.0,
        "gravity": 9.81
    }
}

print(robot_config)


### Access nested values

In [ ]:
robot_name = robot_config["robot"]["name"]
kp = robot_config["controller"]["kp"]
duration = robot_config["simulation"]["duration"]

print("Robot:", robot_name)
print("Kp:", kp)
print("Duration:", duration)


The nested access

```python
robot_config["controller"]["kp"]
```

is comparable to MATLAB:

```matlab
robot_config.controller.kp
```


## 7. Dictionaries and numerical arrays

In [ ]:
state = {
    "position": [0.2, -0.5],
    "velocity": [0.0, 0.1],
    "effort": [1.5, -0.7]
}

print(state["position"])


A dictionary is useful for organizing named groups of data.

However, the values used for numerical computation should usually become NumPy arrays later:

```python
import numpy as np

q = np.array(state["position"])
```

Use dictionaries for organization and configuration.  
Use NumPy arrays for vector and matrix calculations.


## 8. Check whether a key exists

In [ ]:
if "controller" in robot_config:
    print("Controller configuration is available.")

if "payload" not in robot_config["robot"]:
    print("No payload value has been defined.")


## 9. Remove dictionary entries

In [ ]:
temporary_config = {
    "name": "test robot",
    "debug": True,
    "temporary_value": 123
}

removed_value = temporary_config.pop("temporary_value")

print("Removed:", removed_value)
print("Remaining dictionary:", temporary_config)


Other removal methods include:

```python
del temporary_config["debug"]
temporary_config.clear()
```

Use them carefully because they modify the original dictionary.


## 10. Dictionary copy behavior

In [ ]:
original = {
    "controller": {
        "kp": 100.0
    }
}

shallow_copy = original.copy()
shallow_copy["controller"]["kp"] = 200.0

print("Original:", original)
print("Shallow copy:", shallow_copy)


A normal dictionary copy is shallow. Nested objects may still be shared.

Use `copy.deepcopy()` when an independent nested copy is required.


In [ ]:
import copy

original = {
    "controller": {
        "kp": 100.0
    }
}

independent_copy = copy.deepcopy(original)
independent_copy["controller"]["kp"] = 300.0

print("Original:", original)
print("Independent copy:", independent_copy)


## 11. Standard-library file paths with `pathlib`

The `pathlib` module represents files and directories as objects.

This is generally clearer than manually joining path strings.


In [ ]:
from pathlib import Path

data_folder = Path("notebook_04_data")
data_folder.mkdir(exist_ok=True)

json_file = data_folder / "robot_config.json"
csv_file = data_folder / "joint_log.csv"

print("Folder:", data_folder)
print("JSON path:", json_file)
print("CSV path:", csv_file)


The `/` operator joins path components:

```python
data_folder / "robot_config.json"
```

This works across Windows, Linux, and macOS.


## 12. Save a dictionary as JSON

In [ ]:
import json

with json_file.open("w", encoding="utf-8") as file:
    json.dump(robot_config, file, indent=4)

print(f"Saved configuration to: {json_file.resolve()}")


Important ideas:

- `"w"` means write mode.
- `encoding="utf-8"` is a reliable text encoding.
- `with` closes the file automatically.
- `indent=4` makes the JSON readable.


## 13. Read JSON into a dictionary

In [ ]:
with json_file.open("r", encoding="utf-8") as file:
    loaded_config = json.load(file)

print(type(loaded_config))
print(loaded_config)


In [ ]:
print("Loaded robot name:", loaded_config["robot"]["name"])
print("Loaded Kp:", loaded_config["controller"]["kp"])


The complete flow is:

```text
Python dictionary
      ↓ json.dump()
JSON text file
      ↓ json.load()
Python dictionary
```

This is one of the most useful standard-library I/O workflows for robotics software.


## 14. Inspect the JSON text directly

In [ ]:
json_text = json_file.read_text(encoding="utf-8")
print(json_text)


## 15. Modify and save configuration

In [ ]:
loaded_config["controller"]["kp"] = [120.0, 120.0]
loaded_config["robot"]["payload"] = 1.5

updated_json_file = data_folder / "robot_config_updated.json"

with updated_json_file.open("w", encoding="utf-8") as file:
    json.dump(loaded_config, file, indent=4)

print(f"Updated file saved to: {updated_json_file.resolve()}")


## 16. CSV I/O for logged robot data

JSON is useful for configuration. CSV is useful for tabular data such as:

- time
- joint position
- joint velocity
- torque
- sensor measurements


In [ ]:
joint_log = [
    {"time": 0.00, "q1": 0.00, "q2": 0.00},
    {"time": 0.01, "q1": 0.02, "q2": -0.01},
    {"time": 0.02, "q1": 0.04, "q2": -0.02},
    {"time": 0.03, "q1": 0.06, "q2": -0.03}
]

print(joint_log)


This is a list of dictionaries:

- the list represents rows;
- each dictionary represents one row;
- dictionary keys represent column names.


In [ ]:
import csv

fieldnames = ["time", "q1", "q2"]

with csv_file.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(joint_log)

print(f"Saved joint log to: {csv_file.resolve()}")


## 17. Read CSV rows as dictionaries

In [ ]:
with csv_file.open("r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    loaded_rows = list(reader)

print(loaded_rows)


CSV values are read as text by default.

Convert numerical values explicitly.


In [ ]:
numeric_rows = []

for row in loaded_rows:
    numeric_row = {
        "time": float(row["time"]),
        "q1": float(row["q1"]),
        "q2": float(row["q2"])
    }
    numeric_rows.append(numeric_row)

print(numeric_rows)


## 18. Directory inspection with `pathlib`

In [ ]:
print("Files in data folder:")

for path in data_folder.iterdir():
    print(path.name)


### Find only JSON files

In [ ]:
json_files = list(data_folder.glob("*.json"))

for path in json_files:
    print(path.name)


This is useful for robot projects containing multiple configuration files:

```text
config/
    robot.json
    controller.json
    camera.json
    calibration.json
```


## 19. ROS 2 parameter-style configuration

A ROS 2 YAML parameter file often looks conceptually like this:

```yaml
controller_node:
  ros__parameters:
    kp: 100.0
    kd: 20.0
    sampling_time: 0.001
```

The equivalent Python dictionary shape is:


In [ ]:
ros2_parameter_style = {
    "controller_node": {
        "ros__parameters": {
            "kp": 100.0,
            "kd": 20.0,
            "sampling_time": 0.001
        }
    }
}

parameters = ros2_parameter_style["controller_node"]["ros__parameters"]

print(parameters["kp"])
print(parameters["sampling_time"])


A real ROS 2 node does not normally load every parameter by directly parsing a dictionary.  
ROS 2 provides its own parameter API.

However, dictionaries remain useful for understanding the shape of configuration data and for preparing data before passing it into ROS 2 APIs.


## 20. Simulated ROS 2 node configuration

In [ ]:
node_config = {
    "node_name": "joint_controller",
    "publishers": {
        "joint_command": "/joint_command",
        "diagnostics": "/diagnostics"
    },
    "subscribers": {
        "joint_state": "/joint_states"
    },
    "timers": {
        "control_period": 0.001,
        "diagnostic_period": 1.0
    }
}

print("Node name:", node_config["node_name"])
print("Command topic:", node_config["publishers"]["joint_command"])
print("Control period:", node_config["timers"]["control_period"])


This example illustrates an important distinction:

- the dictionary describes the configuration;
- the ROS 2 `Node` object performs the real communication;
- publishers, subscribers, timers, and messages are objects, not merely dictionary entries.


## 21. Function that accepts a configuration dictionary

In [ ]:
def calculate_control_torque(config, position_error, velocity_error):
    kp = config["kp"]
    kd = config["kd"]

    torque = kp * position_error + kd * velocity_error
    return torque


controller = {
    "kp": 100.0,
    "kd": 20.0
}

tau = calculate_control_torque(
    controller,
    position_error=0.05,
    velocity_error=-0.01
)

print("Torque command:", tau)


Configuration dictionaries are convenient function inputs when several related settings must travel together.


## 22. Validate required keys

In [ ]:
def validate_controller_config(config):
    required_keys = ["kp", "kd", "sampling_time"]

    missing_keys = [
        key for key in required_keys
        if key not in config
    ]

    if missing_keys:
        raise ValueError(f"Missing configuration keys: {missing_keys}")

    return True


valid_controller = {
    "kp": 100.0,
    "kd": 20.0,
    "sampling_time": 0.001
}

print(validate_controller_config(valid_controller))


Validation is important because dictionary keys are not automatically checked by Python.

Larger projects may later use:

- classes
- `dataclasses`
- type hints
- configuration validation libraries
- ROS 2 parameter declarations


## 23. Practice exercise 1 — Build a robot dictionary

Create a dictionary named `mobile_robot` containing:

- robot name
- wheel radius
- wheel separation
- maximum velocity
- frame names for `base`, `odom`, and `laser`

Then print the laser frame name.


In [ ]:
# Write your solution here

mobile_robot = {
    # Add your entries
}


## 24. Practice exercise 2 — MATLAB structure conversion

Convert this MATLAB structure into a Python dictionary:

```matlab
controller.name = "computed_torque";
controller.gains.kp = [100 100];
controller.gains.kd = [20 20];
controller.sample_time = 0.001;
```


In [ ]:
# Write your solution here

controller_config = {
    # Add your entries
}


## 25. Practice exercise 3 — Save and load JSON

1. Save `controller_config` as `controller_config.json`.
2. Load the file into `loaded_controller`.
3. Print the loaded proportional gains.


In [ ]:
# Write your solution here


## 26. Practice exercise 4 — Joint-state log

Create a list of dictionaries representing five time samples.

Each row should contain:

- `time`
- `position`
- `velocity`

Save the data to a CSV file.


In [ ]:
# Write your solution here


## 27. Practice exercise 5 — Choose the correct container

Choose between:

- Python list
- Python dictionary
- NumPy array
- Python class/object

for each engineering use:

1. controller gain configuration
2. matrix multiplication
3. ordered list of topic names
4. a ROS 2 node
5. named calibration values
6. a joint-space state vector


In [ ]:
answers = {
    "controller gain configuration": "",
    "matrix multiplication": "",
    "ordered list of topic names": "",
    "ROS 2 node": "",
    "named calibration values": "",
    "joint-space state vector": ""
}

for item, answer in answers.items():
    print(f"{item}: {answer}")


## 28. Suggested answers

Run this section only after attempting the exercises.


In [ ]:
suggested_mobile_robot = {
    "name": "teaching_amr",
    "wheel_radius": 0.05,
    "wheel_separation": 0.30,
    "maximum_velocity": 1.0,
    "frames": {
        "base": "base_link",
        "odom": "odom",
        "laser": "laser_frame"
    }
}

print(suggested_mobile_robot["frames"]["laser"])


In [ ]:
suggested_controller_config = {
    "name": "computed_torque",
    "gains": {
        "kp": [100, 100],
        "kd": [20, 20]
    },
    "sample_time": 0.001
}

print(suggested_controller_config)


In [ ]:
suggested_answers = {
    "controller gain configuration": "dictionary",
    "matrix multiplication": "NumPy array",
    "ordered list of topic names": "list",
    "ROS 2 node": "class/object",
    "named calibration values": "dictionary",
    "joint-space state vector": "NumPy array"
}

for item, answer in suggested_answers.items():
    print(f"{item}: {answer}")


## 29. Summary

### Main comparison

| MATLAB | Python | Typical robotics use |
|---|---|---|
| `struct` | `dict` | configuration and named values |
| cell array | `list` | ordered collections and mixed objects |
| numeric array | NumPy array | vectors, matrices, and computation |
| class/object | class/object | ROS 2 nodes, messages, publishers, and subscribers |

### Main I/O tools

| Tool | Purpose |
|---|---|
| `pathlib.Path` | cross-platform file and folder paths |
| `json` | configuration and nested structured data |
| `csv` | tabular logs and measurements |
| `with ... open(...)` | safe file opening and closing |

### Engineering rule of thumb

Use a dictionary when the names of the values matter.

Use a NumPy array when mathematical operations matter.

Use a list when order matters.

Use a class when behavior and persistent state matter.


## 30. Next notebook

A logical next notebook is:

# Notebook 5 — Python classes and objects for ROS 2 nodes

Topics could include:

- `class`
- `__init__`
- `self`
- object attributes
- object methods
- inheritance
- ROS 2 `Node`
- publishers, subscribers, timers, and callbacks
